In [1]:
# ==============================================================================
# Machine Learning Assignment 2
# Student Name : Fakrudeen Ali
# BITS ID      : 2026AJ05480
# ==============================================================================

import os
import warnings
warnings.filterwarnings("ignore")  # Suppresses FutureWarnings and UserWarnings

import joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score,
    recall_score, f1_score, matthews_corrcoef
)

print("=" * 65)
print("BITS Pilani - M.Tech (AIML) - Machine Learning Assignment 2")
print("Student Name : Fakrudeen Ali")
print("BITS ID      : 2026AJ05480")
print("=" * 65 + "\n")

# 1. Load UCI Wine Quality dataset
red_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
white_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-white.csv"

red = pd.read_csv(red_url, sep=';')
white = pd.read_csv(white_url, sep=';')
red['wine_type'] = 0
white['wine_type'] = 1

data = pd.concat([red, white], ignore_index=True)
data['target'] = (data['quality'] >= 6).astype(int)
data = data.drop(columns=['quality'])

X = data.drop(columns=['target'])
y = data['target']

# 2. Train-Test Split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Export test data
test_df = pd.concat([X_test, y_test], axis=1)
test_df.to_csv("test_data.csv", index=False)

# 3. Fit and save standard scaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

os.makedirs("model", exist_ok=True)
joblib.dump(scaler, "model/scaler.pkl")

# 4. Train 5 classification models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=6, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Naive Bayes": GaussianNB(),
    "Support Vector Machine": SVC(probability=True, random_state=42)
}

results = []
for name, clf in models.items():
    if name in ["Logistic Regression", "KNN", "Support Vector Machine"]:
        clf.fit(X_train_scaled, y_train)
        pred = clf.predict(X_test_scaled)
        proba = clf.predict_proba(X_test_scaled)[:, 1]
    else:
        clf.fit(X_train, y_train)
        pred = clf.predict(X_test)
        proba = clf.predict_proba(X_test)[:, 1]

    save_path = f"model/{name.lower().replace(' ', '_')}.pkl"
    joblib.dump(clf, save_path)

    results.append({
        "ML Model Name": name,
        "Accuracy": round(accuracy_score(y_test, pred), 4),
        "AUC": round(roc_auc_score(y_test, proba), 4),
        "Precision": round(precision_score(y_test, pred, average='weighted', zero_division=0), 4),
        "Recall": round(recall_score(y_test, pred, average='weighted', zero_division=0), 4),
        "F1": round(f1_score(y_test, pred, average='weighted', zero_division=0), 4),
        "MCC": round(matthews_corrcoef(y_test, pred), 4)
    })

results_df = pd.DataFrame(results)
print("=== MODEL EVALUATION METRICS ===")
print(results_df.to_string(index=False))

BITS Pilani - M.Tech (AIML) - Machine Learning Assignment 2
Student Name : Fakrudeen Ali
BITS ID      : 2026AJ05480

=== MODEL EVALUATION METRICS ===
         ML Model Name  Accuracy    AUC  Precision  Recall     F1    MCC
   Logistic Regression    0.7392 0.8057     0.7333  0.7392 0.7329 0.4214
         Decision Tree    0.7500 0.7985     0.7499  0.7500 0.7499 0.4616
                   KNN    0.7408 0.8004     0.7363  0.7408 0.7373 0.4308
           Naive Bayes    0.6831 0.7445     0.6714  0.6831 0.6710 0.2861
Support Vector Machine    0.7731 0.8400     0.7694  0.7731 0.7669 0.4971
